# Background

## CONTEXT

Financial fraud is relevant to HMRC, DWP, FCA, and the Home Office. Traditional rule-based and statistical fraud detectors operate on structured fields — amounts, frequencies, counterparties. But fraud signals are also present in unstructured text: transaction narratives, company filings, communications. LLM-derived semantic features can complement and enrich structured models, but the stakes are high — a false positive has real consequences for the person or business flagged.

***

## ANALYTICAL GOAL

Combine structured financial features with LLM-extracted semantic features from text to improve anomaly detection or fraud classification. Demonstrate that the combined feature set outperforms structured features alone, and that the model's reasoning is interpretable by a human investigator.

***

## DATA

* **IEEE-CIS Fraud Detection dataset (Kaggle):** Structured transaction data with fraud labels — primary dataset
* **Synthetic transaction narratives:** Pre-generated text descriptions for LLM feature extraction exercises
* **Bitcoin / Ethereum public data (BigQuery):** BASD-8 / BABD-13 datasets for crypto transaction graph analysis (see LLM4TO Paper, arXiv:2501.18158)

***

## WORKFLOW & TOOLS

### Workflow stages

* Ingest: Load IEEE-CIS data and synthetic narratives to Delta Lake
* Extract: ai\_extract / ai\_classify for semantic features from transaction text (risk language, entity types, anomalous phrasing)
* Feature engineer: Feature Store for combined semantic + structured feature set; SHAP for feature importance
* Model: MLflow AutoML (SHAP enabled) for fraud classifier; Fairlearn for bias evaluation
* Human review: Databricks Apps investigator review workflow — flags go to human before action
* Govern: Unity Catalog access control; model card; ATRS registration for any operational deployment

### Databricks tools

* ai\_extract, ai\_classify (Agent Bricks)
* Feature Store
* MLflow AutoML, SHAP (shap\_enabled=True)
* Fairlearn (protected characteristics must not drive scores)
* Databricks Apps (investigator UI)
* Unity Catalog lineage + system tables
* MLflow Model Registry (model card, versioning)

***

## AI PLAYBOOK PRINCIPLES

| Principle | Description                           |
| --------- | ------------------------------------- |
| **P1**    | False positive risk is real           |
| **P2**    | Protected chars must not drive scores |
| **P4**    | Investigator reviews before action    |
| **P10**   | Model card + ATRS required            |




Aim: Outperform model with just the structured data. 
Structured Data = Performance X
Structured Data + unstructured data = Performance Y 
we need to show Y is better than X


https://github.com/emmastowell/responsible-ai

https://www.kaggle.com/datasets/lnasiri007/ieeecis-fraud-detection

https://github.com/brtarran/financial_fraud_busters

## IEEE-CIS Data Description
Transaction table “It contains money transfer and also other gifting goods and service, like you booked a ticket for others, etc.”

- TransactionDT: timedelta from a given reference datetime (not an actual timestamp) “TransactionDT first value is 86400, which corresponds to the number of seconds in a day (60 * 60 * 24 = 86400) so I think the unit is seconds. Using this, we know the data spans 6 months, as the maximum value is 15811131, which would correspond to day 183.”

- TransactionAMT: transaction payment amount in USD “Some of the transaction amounts have three decimal places to the right of the decimal point. There seems to be a link to three decimal places and a blank addr1 and addr2 field. Is it possible that these are foreign transactions and that, for example, the 75.887 in row 12 is the result of multiplying a foreign currency amount by an exchange rate?”

- ProductCD: product code, the product for each transaction “Product isn't necessary to be a real 'product' (like one item to be added to the shopping cart). It could be any kind of service.”

- card1 - card6: payment card information, such as card type, card category, issue bank, country, etc.

- addr: address “both addresses are for purchaser addr1 as billing region addr2 as billing country” dist: distance "distances between (not limited) billing address, mailing address, zip code, IP address, phone area, etc.” P_ and (R__) emaildomain: purchaser and recipient email domain “ certain transactions don't need recipient, so R_emaildomain is null.” C1-C14: counting, such as how many addresses are found to be associated with the payment card, etc. The actual meaning is masked. “Can you please give more examples of counts in the variables C1-15? Would these be like counts of phone numbers, email addresses, names associated with the user? I can't think of 15. Your guess is good, plus like device, ipaddr, billingaddr, etc. Also these are for both purchaser and recipient, which doubles the number.” D1-D15: timedelta, such as days between previous transaction, etc. M1-M9: match, such as names on card and address, etc. Vxxx: Vesta engineered rich features, including ranking, counting, and other entity relations. “For example, how many times the payment card associated with a IP and email or address appeared in 24 hours time range, etc.” "All Vesta features were derived as numerical. some of them are count of orders within a clustering, a time-period or condition, so the value is finite and has ordering (or ranking). I wouldn't recommend to treat any of them as categorical. If any of them resulted in binary by chance, it maybe worth trying."

Identity Table Variables in this table are identity information – network connection information (IP, ISP, Proxy, etc) and digital signature (UA/browser/os/version, etc) associated with transactions. They're collected by Vesta’s fraud protection system and digital security partners. (The field names are masked and pairwise dictionary will not be provided for privacy protection and contract agreement)

- DeviceInfo : https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203#583227

- “id01 to id11 are numerical features for identity, which is collected by Vesta and security partners such as device rating, ip_domain rating, proxy rating, etc. Also it recorded behavioral fingerprint like account login times/failed to login times, how long an account stayed on the page, etc. All of these are not able to elaborate due to security partner T&C. I hope you could get basic meaning of these features, and by mentioning them as numerical/categorical, you won't deal with them inappropriately.”

- Labeling logic "The logic of our labeling is define reported chargeback on the card as fraud transaction (isFraud=1) and transactions posterior to it with either user account, email address or billing address directly linked to these attributes as fraud too. If none of above is reported and found beyond 120 days, then we define as legit transaction (isFraud=0). However, in real world fraudulent activity might not be reported, e.g. cardholder was unaware, or forgot to report in time and beyond the claim period, etc. In such cases, supposed fraud might be labeled as legit, but we never could know of them. Thus, we think they're unusual cases and negligible portion." Read more : https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203#588953

# Workflow Plan: Structured + Unstructured Fraud/Anomaly Detection

**Objective:** Demonstrate that combining structured financial features (`train_transaction` + `train_identity`) with unstructured text signals (`synthetic_crypto_fraud_narratives`) improves anomaly/fraud detection over structured features alone.

---

## Data Landscape

| Dataset | Rows | Key Characteristics |
| --- | --- | --- |
| `hackathon.shared_datasets.train_transaction` | 590,540 | 3.5% fraud; 394 features (amounts, card, email, C/D/V/M columns) |
| `hackathon.shared_datasets.train_identity` | 144,233 | Covers ~24% of transactions; device/network identity signals |
| `hackathon.shared_datasets.synthetic_crypto_fraud_narratives` | 1,000 | 60% fraud / 40% legit; free-text narratives + numeric crypto indicators |

---

## UK Government AI Playbook — Principles Mapped to Workflow

This plan integrates the 10 key principles from the [UK Government AI Playbook](https://www.gov.uk/government/publications/ai-playbook-for-the-uk-government/artificial-intelligence-playbook-for-the-uk-government-html).

---

### Phase 0 — Problem Definition & Proportionality
*Playbook Principles: "Start with clear user need", "Be proportionate", "Justify the use of AI"*

- [ ] Define the detection goal: flag suspicious transactions for human review (not auto-block)
- [ ] Document why AI is appropriate: high-dimensional feature space (394+ cols), non-linear fraud patterns, volume too high for manual review
- [ ] Identify proportionality: semi-supervised anomaly detection augments (does not replace) human decision-making
- [ ] Define success metrics: PR-AUC, Recall@FPR=5%, F1 on fraud class; improvement over structured-only baseline

---

### Phase 1 — Data Assessment & Ethics Review
*Playbook Principles: "Use good data", "Be fair", "Consider ethics early"*

- [ ] **Join structured data**: LEFT JOIN `train_transaction` ↔ `train_identity` on `TransactionID` (expect 24% match rate)
- [ ] **Profile missingness**: identity features will be NULL for 76% of transactions — plan imputation/indicator strategy
- [ ] **Assess bias risks**: check if fraud rate differs by `DeviceType`, `card4`/`card6`, `P_emaildomain` — flag any protected-characteristic proxies
- [ ] **Data quality checks**: duplicate TransactionIDs, timestamp integrity, distribution drift between joined vs. unjoined cohorts
- [ ] **Narratives provenance**: document that crypto narratives are synthetic (not real PII); assess whether patterns transfer to the financial fraud domain

---

### Phase 2 — Feature Engineering
*Playbook Principles: "Iterate and learn", "Use data responsibly"*

#### 2a. Structured Features (from `train_transaction` + `train_identity`)
- [ ] Select high-signal columns: `TransactionAmt`, `C1-C14`, `D1-D15`, `V1-V45`, `ProductCD`, `card4/6`, `addr1/2`
- [ ] Identity enrichment: `DeviceType`, `DeviceInfo`, `id_01`–`id_38` (where available)
- [ ] Derived features: amt-to-card-mean ratio, velocity features (txns per card in last N), email domain risk scoring
- [ ] Handle categoricals: one-hot or target-encoding for `ProductCD`, `card4`, `card6`, email domains
- [ ] Handle missingness: "is_missing" indicator columns + median imputation for numerics

#### 2b. Unstructured Features (from `synthetic_crypto_fraud_narratives`)
- [ ] **Text embeddings**: generate vector representations of `narrative` field using Databricks Foundation Model API (`ai_query` with an embedding model) or sentence-transformers
- [ ] **NLP-derived signals**: extract keyword indicators ("coordinated", "liquidity withdrawn", "newly active wallets"), sentiment polarity, narrative length, entity counts
- [ ] **Structured crypto indicators**: `price_change_pct`, `volume_spike_x`, `social_mention_spike_x`, `top10_holder_pct`, `unique_buy_wallets`
- [ ] Dimensionality reduction on embeddings: PCA or UMAP to ~50 components

---

### Phase 3 — Modelling (Iterative, with Baselines)
*Playbook Principles: "Start small and iterate", "Test rigorously"*

#### Experiment A — Structured-Only Baseline
- [ ] Train XGBoost/LightGBM on joined `train_transaction` + `train_identity` features
- [ ] Address class imbalance: `scale_pos_weight`, SMOTE, or focal loss
- [ ] Log to MLflow: hyperparameters, metrics (ROC-AUC, PR-AUC, Fraud-F1, Recall@5%FPR)

#### Experiment B — Unstructured-Only (Crypto Narratives)
- [ ] Train classifier on `synthetic_crypto_fraud_narratives` using text embeddings + numeric crypto features
- [ ] Establishes whether narrative text alone carries discriminatory power
- [ ] Log to same MLflow experiment for comparison

#### Experiment C — Combined / Augmented Model
- [ ] **Strategy 1 — Feature fusion**: concatenate structured features with text-derived features for the crypto dataset
- [ ] **Strategy 2 — Transfer learning**: train a text-based fraud-pattern detector on narratives, then use its confidence scores as an auxiliary feature in the main structured model
- [ ] **Strategy 3 — Anomaly ensemble**: train isolation forest on structured data + separately on embeddings; combine anomaly scores
- [ ] Compare all experiments on held-out test set; use statistical significance testing (McNemar's or DeLong's test on AUCs)

---

### Phase 4 — Explainability & Transparency
*Playbook Principles: "Be transparent", "Explain decisions", "Maintain human oversight"*

- [ ] SHAP values for structured model — identify top risk drivers per prediction
- [ ] Attention/token-importance for text features — which narrative phrases trigger fraud signals
- [ ] Generate human-readable "risk cards" for flagged transactions (top 3 reasons + confidence)
- [ ] Define escalation thresholds: high-confidence → auto-flag; medium → analyst queue; low → pass

---

### Phase 5 — Fairness & Bias Testing
*Playbook Principles: "Be fair", "Consider and address potential harms"*

- [ ] Evaluate model performance across demographic proxies (`DeviceType`, `card4`, email domain categories)
- [ ] Check for disparate false-positive rates across groups
- [ ] Document any fairness trade-offs and mitigation steps taken
- [ ] Test on the crypto narratives: does model equally detect different fraud typologies (pump-and-dump, liquidity rug, coordinated sell)?

---

### Phase 6 — Governance, Accountability & Deployment Readiness
*Playbook Principles: "Be accountable", "Manage risks", "Use AI legally"*

- [ ] Register final model in Unity Catalog Model Registry with version, lineage, and metadata tags
- [ ] Document model card: intended use, limitations, known biases, performance boundaries
- [ ] Define monitoring plan: data drift detection, performance decay alerts, retraining triggers
- [ ] RACI matrix: who approves model updates, who reviews flagged cases, who owns data quality
- [ ] Audit trail: MLflow experiment tracking provides full reproducibility of all decisions

---

### Phase 7 — Share, Collaborate & Iterate
*Playbook Principles: "Share and collaborate", "Build internal capability"*

- [ ] Publish findings to team dashboard (structured-only vs. combined performance comparison)
- [ ] Document lessons learned: what worked, what didn't, where unstructured data added most value
- [ ] Propose next iteration: real-time scoring pipeline, additional unstructured sources (transaction memos, customer comms)

---

## Expected Outcome

| Metric | Structured Only (X) | Structured + Unstructured (Y) |
| --- | --- | --- |
| ROC-AUC | Baseline | Target: +2-5% |
| PR-AUC | Baseline | Target: +5-10% |
| Fraud Recall @5% FPR | Baseline | Target: +10-15% |
| Explainability | Feature importance only | Feature importance + narrative reasoning |

**Hypothesis:** Unstructured signals capture fraud *intent* and *coordination* patterns that structured features alone cannot encode, particularly for novel/emerging fraud typologies.

# Ingest

### What we are doing and why

The first stage of every pipeline is getting your raw data into **Delta Lake** under Unity Catalog governance, having embedded data protection in the assets and having provided metadata. This matters for four reasons that map directly to the AI Playbook:

- **Principle 2:** You need to know the data you are working with and determine if there are any special legal considerations for working with it. Is the data sensitive? Is there personal data? What limitations or biases may the data have that will need to be balanced for in modelling or limit the situations in which the tool can suitably be used.
- **Principle 4:** Starting out with a manual investigation of the data provides human control at the very start of the process, and permits first considerations of tooling costs.
- **Principle 7:** By providing metadata when saving tables, it is possible for other users to discover and understand the datasets you create and how they were generated.
- **Principle 10:** Unity Catalog records who created each table, when, and from what source and tracks the lineage of data assets. This is the first stage in assurance and audit of AI systems.

In [0]:
# import code from tutorial
import re
import os
from datetime import datetime, UTC
from typing import Optional
import json

import pandas as pd
import numpy as np

from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, FloatType, BooleanType
)


import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# read in tables
train_tx = spark.table("hackathon.shared_datasets.train_transaction")
train_tx.display()

## Exploratory Data Analysis

In [0]:
# ── 1. Dataset Overview & Class Imbalance ─────────────────────────────────────
n_rows = train_tx.count()
n_cols = len(train_tx.columns)

fraud_counts = train_tx.groupBy("isFraud").count().orderBy("isFraud").toPandas()
fraud_counts["label"] = fraud_counts["isFraud"].map({0: "Legit", 1: "Fraud"})
n_fraud = int(fraud_counts.loc[fraud_counts["isFraud"] == 1, "count"].values[0])
fraud_rate = n_fraud / n_rows * 100

print(f"Dataset: {n_rows:,} rows x {n_cols} columns")
print(f"  Fraud : {n_fraud:,}  ({fraud_rate:.2f}%)")
print(f"  Legit : {n_rows - n_fraud:,}  ({100 - fraud_rate:.2f}%)")
print(f"\n  Class imbalance ratio  1 : {(n_rows - n_fraud) / n_fraud:.0f}")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(fraud_counts["label"], fraud_counts["count"],
             color=["steelblue", "crimson"], edgecolor="white", width=0.5)
ax.set_title("IEEE-CIS — Class Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Transaction Count")
for bar, val, pct in zip(bars, fraud_counts["count"],
                          fraud_counts["count"] / n_rows * 100):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}\n({pct:.2f}%)", ha="center", fontsize=10)
ax.set_ylim(0, max(fraud_counts["count"]) * 1.15)
plt.tight_layout()
plt.show()

In [0]:
# ── 2. Missing Value Analysis (single-pass over all columns) ──────────────────
null_exprs = []
for c, dtype in train_tx.dtypes:
    cond = (F.col(c).isNull() | F.isnan(F.col(c))) if dtype in ("double", "float") else F.col(c).isNull()
    null_exprs.append((F.mean(F.when(cond, 1).otherwise(0)) * 100).alias(c))

null_pcts = train_tx.agg(*null_exprs).toPandas()
null_df = null_pcts.T.reset_index()
null_df.columns = ["column", "null_pct"]
null_df = null_df.sort_values("null_pct", ascending=False).reset_index(drop=True)

print(f"Columns with  >0% missing : {(null_df['null_pct']  > 0).sum()}")
print(f"Columns with >50% missing : {(null_df['null_pct'] > 50).sum()}")
print(f"Columns with >80% missing : {(null_df['null_pct'] > 80).sum()}")
print(f"Columns with   0% missing : {(null_df['null_pct'] == 0).sum()}")

top_null = null_df[null_df["null_pct"] > 0].head(40)
plt.figure(figsize=(16, 5))
plt.bar(range(len(top_null)), top_null["null_pct"], color="coral", edgecolor="white")
plt.xticks(range(len(top_null)), top_null["column"], rotation=90, fontsize=8)
plt.axhline(50, color="red",     linestyle="--", alpha=0.7, label="50% threshold")
plt.axhline(80, color="darkred", linestyle="--", alpha=0.7, label="80% threshold")
plt.title("Top 40 Columns by Missing Value %", fontsize=13)
plt.ylabel("Missing %")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# ── 3. Transaction Amount Distribution ────────────────────────────────────────
amt_pdf = train_tx.select("TransactionAmt", "isFraud").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, "steelblue", "Legit"), (1, "crimson", "Fraud")]:
    subset = amt_pdf[amt_pdf["isFraud"] == label]["TransactionAmt"]
    axes[0].hist(np.log1p(subset), bins=60, alpha=0.6, density=True,
                 label=f"{name} (n={len(subset):,})", color=color)
axes[0].set_title("TransactionAmt Distribution (log scale)")
axes[0].set_xlabel("log(1 + Amount)")
axes[0].set_ylabel("Density")
axes[0].legend()

p95 = amt_pdf["TransactionAmt"].quantile(0.95)
bp = axes[1].boxplot(
    [amt_pdf[amt_pdf["isFraud"] == 0]["TransactionAmt"].clip(upper=p95),
     amt_pdf[amt_pdf["isFraud"] == 1]["TransactionAmt"].clip(upper=p95)],
    labels=["Legit", "Fraud"], patch_artist=True
)
for patch, color in zip(bp["boxes"], ["steelblue", "crimson"]):
    patch.set_facecolor(color)
axes[1].set_title(f"Boxplot (clipped at 95th pct: ${p95:.0f})")
axes[1].set_ylabel("Amount ($)")

plt.suptitle("Transaction Amount Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\nTransactionAmt stats by class:")
print(amt_pdf.groupby("isFraud")["TransactionAmt"].describe()
      .rename(index={0: "Legit", 1: "Fraud"}).round(2).to_string())

In [0]:
# ── 4. Categorical Feature Fraud Rates ────────────────────────────────────────
cat_features = ["ProductCD", "card4", "card6"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col_name in enumerate(cat_features):
    cat_df = (train_tx
              .filter(F.col(col_name).isNotNull())
              .groupBy(col_name, "isFraud").count()
              .toPandas())
    pivot = cat_df.pivot(index=col_name, columns="isFraud", values="count").fillna(0)
    legit = pivot.get(0, pd.Series(0, index=pivot.index))
    fraud = pivot.get(1, pd.Series(0, index=pivot.index))
    total = legit + fraud
    fraud_rate = (fraud / total * 100).sort_values(ascending=False)

    axes[i].bar(fraud_rate.index.astype(str), fraud_rate.values,
                color="crimson", edgecolor="white")
    axes[i].set_title(f"Fraud Rate by {col_name}")
    axes[i].set_ylabel("Fraud Rate (%)")
    axes[i].tick_params(axis="x", rotation=45)
    for j, (fr, tot) in enumerate(zip(fraud_rate.values, total.loc[fraud_rate.index].values)):
        axes[i].text(j, fr + 0.1, f"{fr:.1f}%\n(n={int(tot):,})", ha="center", fontsize=7)

plt.suptitle("Fraud Rate by Categorical Features", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# ── 5. Temporal Patterns (TransactionDT = seconds offset from reference date) ──
tx_time = train_tx.select(
    "isFraud",
    (F.col("TransactionDT") % 86400 / 3600).cast("int").alias("hour"),
    (F.col("TransactionDT") / 86400).cast("int").alias("day")
).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

hourly = tx_time.groupby("hour")["isFraud"].agg(["sum", "count"])
hourly["fraud_rate"] = hourly["sum"] / hourly["count"] * 100
axes[0].bar(hourly.index, hourly["fraud_rate"], color="steelblue", edgecolor="white")
axes[0].set_title("Fraud Rate by Hour of Day (relative)")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_xticks(range(24))

daily = tx_time.groupby("day")["isFraud"].agg(["sum", "count"])
daily["fraud_rate"] = daily["sum"] / daily["count"] * 100
ax2 = axes[1].twinx()
axes[1].bar(daily.index, daily["count"], color="steelblue", alpha=0.35, label="Volume")
ax2.plot(daily.index, daily["fraud_rate"], color="crimson", linewidth=1.5, label="Fraud Rate %")
axes[1].set_title("Daily Volume & Fraud Rate (relative time)")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Transaction Count")
ax2.set_ylabel("Fraud Rate (%)")
l1, lab1 = axes[1].get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
axes[1].legend(l1 + l2, lab1 + lab2, loc="upper left")

plt.suptitle("Temporal Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# ── 6. Count Features (C-columns) & Email Domain Fraud Rates ──────────────────
c_cols = [f"C{i}" for i in range(1, 15)]

c_pdf = train_tx.select(c_cols + ["isFraud"]).toPandas()
c_medians = c_pdf.groupby("isFraud")[c_cols].median().T
c_medians.columns = ["Legit", "Fraud"]
c_medians["ratio"] = c_medians["Fraud"] / c_medians["Legit"].replace(0, np.nan)
c_medians = c_medians.sort_values("ratio", ascending=False)

# Single-pass aggregation avoids the extra shuffle from the previous two-step join
email_df = (
    train_tx
    .filter(F.col("P_emaildomain").isNotNull())
    .groupBy("P_emaildomain")
    .agg(
        F.count("*").alias("total"),
        (F.sum(F.col("isFraud").cast("double")) / F.count("*") * 100).alias("fraud_rate"),
    )
    .filter(F.col("total") > 100)
    .orderBy(F.col("fraud_rate").desc())
    .limit(12)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

c_medians[["Legit", "Fraud"]].plot(kind="bar", ax=axes[0],
                                    color=["steelblue", "crimson"], edgecolor="white")
axes[0].set_title("Median C-Feature Values by Class")
axes[0].set_ylabel("Median Value")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()

axes[1].bar(email_df["P_emaildomain"], email_df["fraud_rate"],
            color="darkorange", edgecolor="white")
axes[1].set_title("Top Email Domains by Fraud Rate (>=100 txns)")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].tick_params(axis="x", rotation=45, labelsize=9)

plt.suptitle("Count Features & Email Domain Analysis", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# ── Structured Data Baseline (scikit-learn + MLflow) ─────────────────────────
# Using scikit-learn avoids the Spark Connect 256 MB model-size limit that
# fires when fitting large PySpark ML StringIndexer vocabularies.

import mlflow
from mlflow.models import infer_signature

from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report,
    average_precision_score, precision_score, recall_score, f1_score,
)

# ── Feature selection ─────────────────────────────────────────────────────────
numeric_cols = ["TransactionAmt", "TransactionDT"] + [f"C{i}" for i in range(1, 15)]
cat_cols     = ["ProductCD", "card4", "card6", "P_emaildomain"]
label_col    = "isFraud"

pdf = train_tx.select(numeric_cols + cat_cols + [label_col]).toPandas()
X   = pdf[numeric_cols + cat_cols]
y   = pdf[label_col].astype(int)

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# ── Preprocessing ─────────────────────────────────────────────────────────────
cat_transformer = SkPipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
    ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
preprocessor = ColumnTransformer([
    ("num", "passthrough", numeric_cols),  # HistGBM handles NaN natively
    ("cat", cat_transformer, cat_cols),
], remainder="drop")

# ── MLflow experiment ─────────────────────────────────────────────────────────
mlflow.set_experiment("/Users/labuser15808758_1783593571@vocareum.com/Financial_Fraud_Busters")

model_params = {"max_iter": 200, "max_depth": 6, "random_state": 42}

with mlflow.start_run(run_name="baseline_structured_only") as run:

    # ── Params ────────────────────────────────────────────────────────────────
    mlflow.log_params({
        **model_params,
        "model_type":        "HistGradientBoostingClassifier",
        "n_numeric_features": len(numeric_cols),
        "n_cat_features":    len(cat_cols),
        "train_size":        len(X_train),
        "test_size":         len(X_test),
        "features":          numeric_cols + cat_cols,
    })

    # ── Build & fit ───────────────────────────────────────────────────────────
    baseline_model = SkPipeline([
        ("prep", preprocessor),
        ("clf",  HistGradientBoostingClassifier(**model_params)),
    ])
    print("Training baseline model (structured features only) …")
    baseline_model.fit(X_train, y_train)

    # ── Metrics ───────────────────────────────────────────────────────────────
    y_proba = baseline_model.predict_proba(X_test)[:, 1]
    y_pred  = baseline_model.predict(X_test)

    auc_baseline = roc_auc_score(y_test, y_proba)
    avg_prec     = average_precision_score(y_test, y_proba)
    fraud_f1     = f1_score(y_test, y_pred, pos_label=1)
    fraud_prec   = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    fraud_recall = recall_score(y_test, y_pred, pos_label=1)

    mlflow.log_metrics({
        "roc_auc":         auc_baseline,
        "avg_precision":   avg_prec,
        "fraud_f1":        fraud_f1,
        "fraud_precision": fraud_prec,
        "fraud_recall":    fraud_recall,
    })

    # ── Feature importances ──────────────────────────────────────────────────────────
    feature_names = numeric_cols + cat_cols
    imp_series = pd.Series(
        baseline_model.named_steps["clf"].feature_importances_,
        index=feature_names,
    ).sort_values(ascending=True)

    fig_imp, ax_imp = plt.subplots(figsize=(8, 6))
    imp_series.plot(kind="barh", ax=ax_imp, color="steelblue", edgecolor="white")
    ax_imp.set_title("Feature Importances — Baseline Model", fontsize=12, fontweight="bold")
    ax_imp.set_xlabel("Importance")
    plt.tight_layout()
    mlflow.log_figure(fig_imp, "feature_importance.png")
    plt.show()
    plt.close(fig_imp)

    # ── Log model ─────────────────────────────────────────────────────────────
    signature = infer_signature(X_train, y_proba)
    mlflow.sklearn.log_model(
        baseline_model,
        name="fraud_baseline",
        signature=signature,
        input_example=X_train[:5],
    )

    run_id = run.info.run_id

print(f"\nMLflow run ID : {run_id}")
print(f"ROC-AUC       : {auc_baseline:.4f}  (baseline — structured only)")
print(f"Avg Precision : {avg_prec:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

# Ingest - unstructured data

The table contains narratives related to synthetic cryptocurrency fraud incidents. It includes information such as the type of fraud (label), the detailed narrative of the incident, price changes, social media mention spikes, and other relevant metrics. Possible use cases include analyzing trends in cryptocurrency fraud, understanding the impact of social mentions on price movements, and investigating unique wallet activity associated with these incidents.



In [0]:
narratives_df = spark.table("hackathon.shared_datasets.synthetic_crypto_fraud_narratives")
display(narratives_df)

## Bitcoin data

https://arxiv.org/html/2501.18158v1



Unpacks a crypto transaction-subgraph archive into a shared Volume and builds summary and manifest tables (GraphML subgraphs plus LLM4TG text representations).

In [0]:


crypto_summary = spark.table("hackathon.shared_datasets.crypto_subgraph_summary")
crypto_manifest = spark.table("hackathon.shared_datasets.crypto_subgraph_manifest")

display(crypto_summary)
display(crypto_manifest)

# Extract

# Feature engineer

# Model

# Human review

# Govern